# Random-address PRF capture: SLH-DSA with ChipWhisperer

This notebook captures independent WOTS+/FORS address-PRF invocations using
randomly selected valid addresses. It supports all six SLH-DSA parameter sets
and compiler options `O0`, `O1`, `O2`, `O3`, and `Os`.

For each condition: select parameter header, build firmware, program target,
calibrate acquisition, and capture across multiple **experimental** secret seeds.
Completed conditions and per-seed data can be resumed using the saved manifests.

**Important:** The PRF-only SimpleSerial firmware is included in the `firmware/simpleserial-SLH-DSA/` directory.
Copy it into the ChipWhisperer firmware source tree before building.
The example seeds are public and insecure by design; do not use them for real cryptographic keys.


## 1. Imports and local paths

Configure `SLH_DSA_FIRMWARE_DIR` and optionally `SLH_DSA_RESULTS_DIR` in your local environment.


In [ ]:
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

import chipwhisperer as cw

# Run this notebook from the repository root or the notebooks/ folder.
WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = WORKDIR if (WORKDIR / "src").is_dir() else WORKDIR.parent
MODULE_DIR = PROJECT_ROOT / "src"
if not MODULE_DIR.is_dir():
    raise FileNotFoundError("Cannot find src/. Start Jupyter from the repository.")
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

# Override these paths locally via environment variables (do not commit paths).
FIRMWARE_DIR = Path(os.environ.get(
    "SLH_DSA_FIRMWARE_DIR",
    str(PROJECT_ROOT / "firmware" / "simpleserial-SLH-DSA"),
)).expanduser()
RESULTS_BASE_DIR = Path(os.environ.get(
    "SLH_DSA_RESULTS_DIR", str(PROJECT_ROOT / "results"),
)).expanduser()

import prf_capture_core as live
live = importlib.reload(live)

import random_address_capture as autobatch
autobatch = importlib.reload(autobatch)

print("Capture helper:", live.CAPTURE_HELPER_VERSION)
print("Batch helper:", autobatch.BATCH_HELPER_VERSION)
print("Firmware directory:", FIRMWARE_DIR)
print("Results directory:", RESULTS_BASE_DIR)


## 2. Capture configuration

Use the small default experiment first. Increase counts and select all parameter sets and compiler options only after validating hardware and storage.


In [ ]:
SCOPETYPE = "OPENADC"
PLATFORM = "CW308_STM32F3"
CRYPTO_TARGET = "MBEDTLS"
HASH_MODE = "sha2"

# Safe, small example. For the full matrix, enable all six sets and five opts.
PARAM_SETS = ["128f"]
COMPILER_OPTS = ["Os"]
BUILD_FIRMWARE = True
PROGRAM_TARGET = True

# Demonstration settings (adjust for your own experiment).
NUM_SECRET_SEEDS = 3
CPA_RANDOM_TRACES_PER_SECRET_SEED = 100
SECRET_SEED_MASTER = int(os.environ.get("SLH_DSA_SECRET_SEED_MASTER", "424242"))
CAPTURE_RNG_SEED = 1234
MULTI_SEED_CAPTURE_KIND = "both"  # "wots", "fors", or "both"
SHUFFLE_FIXED_RANDOM_SEQUENCE = True

# Optional TVLA: first secret seed of the Os build only.
CAPTURE_TVLA_ON_OS_FIRST_SEED = False
TVLA_FIXED_TRACES = 10000
TVLA_RANDOM_TRACES = 10000
TVLA_SEED_INDEX = 0

# Restart/resume controls. A mismatched existing secret pool is rejected.
RESUME = True
COMPATIBLE_RESUME = True
IGNORE_EXISTING_FIXED_NONPRIMARY = True
REGENERATE_SHARED_SEED_MANIFEST = False
CONTINUE_ON_ERROR = False

CALIBRATION_KIND = "wots"
numberOfPoints = -1
MAX_SAMPLES = 131124
AUTO_CALIBRATE_POINTS = True
CALIBRATION_MARGIN = 0
presamples = 0
gain = 15
capture_timeout_s = 10
ss_timeout = 250
VERIFY_ADDR_EACH_TIME = False
VERIFY_RESULT = True
VERIFY_FIRST_N = 5
FLUSH_EVERY = 1000

print("Number of capture conditions:", len(PARAM_SETS) * len(COMPILER_OPTS))


## 3. Preview planned captures


In [ ]:
plan = autobatch.preview_capture_plan(
    param_sets=PARAM_SETS,
    compiler_opts=COMPILER_OPTS,
    num_secret_seeds=NUM_SECRET_SEEDS,
    capture_kind=MULTI_SEED_CAPTURE_KIND,
    cpa_random_traces_per_secret_seed=CPA_RANDOM_TRACES_PER_SECRET_SEED,
    capture_tvla_on_os_first_seed=CAPTURE_TVLA_ON_OS_FIRST_SEED,
    tvla_fixed_traces=TVLA_FIXED_TRACES,
    tvla_random_traces=TVLA_RANDOM_TRACES,
)

total_traces = sum(row["planned_traces"] for row in plan)

print("conditions    :", len(plan))
print("planned traces:", f"{total_traces:,}")
print()
for row in plan:
    print(
        f"{row['parameter_set']:4s} / {row['compiler_opt']:2s} : "
        f"{row['planned_traces']:,} traces"
    )


## 4. Connect the ChipWhisperer scope and target


In [ ]:
scope = cw.scope()
scope.default_setup()
scope.gain.db = gain
scope.adc.samples = int(MAX_SAMPLES)
scope.adc.presamples = int(presamples)
scope.adc.timeout = int(capture_timeout_s)

target = cw.target(scope, cw.targets.SimpleSerial)

print("scope/target connected")
print("gain          :", scope.gain.db)
print("initial sample:", scope.adc.samples)


## 5. Run the capture batch

The batch helper checks progress files and resumes complete seed datasets when possible.


In [ ]:
batch_state = autobatch.run_all_parameter_compiler_captures(
    scope,
    target,
    live,

    firmware_dir=FIRMWARE_DIR,
    results_base_dir=RESULTS_BASE_DIR,

    platform=PLATFORM,
    crypto_target=CRYPTO_TARGET,
    hash_mode=HASH_MODE,

    param_sets=PARAM_SETS,
    compiler_opts=COMPILER_OPTS,

    build_firmware=BUILD_FIRMWARE,
    program_target=PROGRAM_TARGET,

    num_secret_seeds=NUM_SECRET_SEEDS,
    cpa_random_traces_per_secret_seed=(
        CPA_RANDOM_TRACES_PER_SECRET_SEED
    ),
    capture_kind=MULTI_SEED_CAPTURE_KIND,

    secret_seed_master=SECRET_SEED_MASTER,
    capture_rng_seed=CAPTURE_RNG_SEED,
    shuffle_fixed_random_sequence=(
        SHUFFLE_FIXED_RANDOM_SEQUENCE
    ),

    capture_tvla_on_os_first_seed=(
        CAPTURE_TVLA_ON_OS_FIRST_SEED
    ),
    tvla_fixed_traces=TVLA_FIXED_TRACES,
    tvla_random_traces=TVLA_RANDOM_TRACES,
    tvla_seed_index=TVLA_SEED_INDEX,

    regenerate_shared_seed_manifest=(
        REGENERATE_SHARED_SEED_MANIFEST
    ),

    calibration_kind=CALIBRATION_KIND,
    number_of_points=numberOfPoints,
    max_samples=MAX_SAMPLES,
    auto_calibrate_points=AUTO_CALIBRATE_POINTS,
    calibration_margin=CALIBRATION_MARGIN,

    timeout=ss_timeout,
    verify_addr_each_time=VERIFY_ADDR_EACH_TIME,
    verify_result=VERIFY_RESULT,
    verify_first_n=VERIFY_FIRST_N,
    flush_every=FLUSH_EVERY,

    resume=RESUME,
    compatible_resume=COMPATIBLE_RESUME,
    ignore_existing_fixed_nonprimary=(
        IGNORE_EXISTING_FIXED_NONPRIMARY
    ),
    continue_on_error=CONTINUE_ON_ERROR,
)

print("batch finished")


## 6. Resume and output

Completed parameter-set/optimization conditions are skipped. Partially finished conditions are rebuilt/programmed, then the low-level helper resumes using its per-seed manifests.

Output layout: `results/SLH_DSA_<param>_sha2_compiler_compare/<compiler>/...`.

Keep `shared_secret_seeds.json` private because it contains experimental secret key material. Never upload output data to a public repository without an independent review.


In [ ]:
# Optional disconnect after all captures finish:
# target.dis()
# scope.dis()
